In [1]:
import sys
print("Python version:", sys.version)
print("Python path:", sys.executable)

Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Python path: c:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\.venv\Scripts\python.exe


## Step 1 — Set the project root in VS Code (choose `.venv` notebook kernel)

In [2]:
from pathlib import Path
import os, sys
ROOT=Path.cwd()
if not (ROOT/'app').is_dir(): ROOT=ROOT.parent
if not (ROOT/'app').is_dir(): raise FileNotFoundError('Open iMAGE_Notebook_Edition as your VS Code folder')
os.chdir(ROOT);sys.path.insert(0,str(ROOT))
print('Project root:',ROOT)

Project root: c:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final


In [ ]:

import torch
import sys

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
    print("Status: CUDA GPU ready for training!")
else:
    DEVICE = torch.device("cpu")
    print("Status: CUDA GPU not detected.")
    print("Current device: CPU")


Python: 3.11.9
PyTorch: 2.15.0+cpu
Status: CUDA GPU not detected.
Current device: CPU


In [1]:

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))

    x = torch.randn(1000, 1000, device=device)
    result = x @ x

    print("GPU computation successful!")
    print("Result device:", result.device)
else:
    print("GPU acceleration unavailable.")


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4070
GPU computation successful!
Result device: cuda:0


In [2]:

from pathlib import Path

# Locate the project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Create dataset folders
folders = [
    "data/raw/SIDD",
    "data/raw/NIND",
    "data/raw/PolyU",
    "data/inventory",
    "data/processed",
]

for folder in folders:
    path = PROJECT_ROOT / folder
    path.mkdir(parents=True, exist_ok=True)
    print(f"Ready: {path}")

print("\nDataset directories created successfully!")


Ready: c:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\data\raw\SIDD
Ready: c:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\data\raw\NIND
Ready: c:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\data\raw\PolyU
Ready: c:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\data\inventory
Ready: c:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\data\processed

Dataset directories created successfully!


In [3]:

%pip install -U huggingface_hub tqdm pillow


   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   ------------------------ --------------- 524.3/857.0 kB 5.7 MB/s eta 0:00:01
   ---------------------------------------- 857.0/857.0 kB 6.4 MB/s  0:00:00
   ---------------------------------------- 0.0/7.3 MB ? eta -:--:--
   -------------- ------------------------- 2.6/7.3 MB 12.5 MB/s eta 0:00:01
   ------------------------------ --------- 5.5/7.3 MB 13.4 MB/s eta 0:00:01
   ---------------------------------------- 7.3/7.3 MB 14.0 MB/s  0:00:00

  Attempting uninstall: pillow

    Found existing installation: pillow 12.3.0

    Uninstalling pillow-12.3.0:

      Successfully uninstalled pillow-12.3.0

   -------- ------------------------------- 1/5 [pillow]
   -------- ------------------------------- 1/5 [pillow]
   -------- ------------------------------- 1/5 [pillow]
   -------- ------------------------------- 1/5 [pillow]
   -------- ------------------------------- 1/5 [pillow]
   -------- ------------------

  You can safely remove it manually.


In [4]:

from pathlib import Path
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm
import zipfile
import shutil

# Locate project root
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)

REPO = "VCLab-PolyU/GGT-100K"

DATASETS = {
    "SIDD": {
        "filename": "Existing-dataset/SIDD.zip",
        "size": "Approximately 2.67 GB"
    },
    "PolyU": {
        "filename": "Existing-dataset/PolyU-Real-World-Noisy-Images-Dataset.zip",
        "size": "Approximately 400 MB"
    }
}

# Change to ["PolyU", "SIDD"] to download both.
# Starting with PolyU is recommended for testing.
SELECTED = ["PolyU"]

def safe_extract(archive, destination):
    destination = destination.resolve()

    with zipfile.ZipFile(archive) as z:
        members = z.infolist()

        for member in members:
            target = (destination / member.filename).resolve()

            if not target.is_relative_to(destination):
                raise ValueError("Unsafe ZIP path detected")

        for member in tqdm(members, desc="Extracting"):
            z.extract(member, destination)

for name in SELECTED:
    info = DATASETS[name]
    destination = DATA_DIR / name
    destination.mkdir(parents=True, exist_ok=True)

    print(f"\nDownloading {name}")
    print("Archive size:", info["size"])

    archive = hf_hub_download(
        repo_id=REPO,
        filename=info["filename"],
        repo_type="dataset"
    )

    print("Extracting:", name)
    safe_extract(archive, destination)

    print(f"{name} extracted into: {destination}")

print("\nSelected dataset downloads completed!")


c:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Archive size: Approximately 400 MB


c:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rmclo\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\.venv\Lib\site-packages\huggingface_hub\file_dow

Extracting: PolyU


Extracting: 100%|██████████| 8004/8004 [00:03<00:00, 2095.59it/s]

PolyU extracted into: c:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\data\raw\PolyU

Selected dataset downloads completed!


In [5]:

from collections import Counter

for name in SELECTED:
    folder = DATA_DIR / name

    files = [
        p for p in folder.rglob("*")
        if p.is_file()
    ]

    extensions = Counter(
        p.suffix.lower() for p in files
    )

    print(f"\n{name}")
    print("Total files:", len(files))
    print("File types:", dict(extensions))

    print("\nSample filenames:")
    for path in files[:10]:
        print(" ", path.relative_to(folder))



PolyU
Total files: 8000
File types: {'.jpg': 8000}

Sample filenames:
  PolyU-Real-World-Noisy-Images-Dataset\train_sub\GT\Canon5D2_bag_s001.jpg
  PolyU-Real-World-Noisy-Images-Dataset\train_sub\GT\Canon5D2_bag_s002.jpg
  PolyU-Real-World-Noisy-Images-Dataset\train_sub\GT\Canon5D2_bag_s003.jpg
  PolyU-Real-World-Noisy-Images-Dataset\train_sub\GT\Canon5D2_bag_s004.jpg
  PolyU-Real-World-Noisy-Images-Dataset\train_sub\GT\Canon5D2_bag_s005.jpg
  PolyU-Real-World-Noisy-Images-Dataset\train_sub\GT\Canon5D2_bag_s006.jpg
  PolyU-Real-World-Noisy-Images-Dataset\train_sub\GT\Canon5D2_bag_s007.jpg
  PolyU-Real-World-Noisy-Images-Dataset\train_sub\GT\Canon5D2_bag_s008.jpg
  PolyU-Real-World-Noisy-Images-Dataset\train_sub\GT\Canon5D2_bag_s009.jpg
  PolyU-Real-World-Noisy-Images-Dataset\train_sub\GT\Canon5D2_bag_s010.jpg


In [6]:

from pathlib import Path
from collections import Counter

polyu_root = DATA_DIR / "PolyU"

print("POLYU DATASET STRUCTURE")
print("=" * 50)

# List all folders containing JPG images
for folder in sorted(polyu_root.rglob("*")):
    if not folder.is_dir():
        continue

    jpg_count = sum(
        1 for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() == ".jpg"
    )

    if jpg_count > 0:
        print(f"{folder.relative_to(polyu_root)}: {jpg_count} images")

# Count all images
all_images = list(polyu_root.rglob("*.jpg"))

print("\nTotal JPG images:", len(all_images))

# Display sample filenames from each parent folder
print("\nEXAMPLE FILENAMES")
for folder in sorted(set(p.parent for p in all_images)):
    print(f"\n{folder.relative_to(polyu_root)}")
    for img in sorted(folder.glob("*.jpg"))[:3]:
        print("  ", img.name)



POLYU DATASET STRUCTURE
PolyU-Real-World-Noisy-Images-Dataset\train_sub\GT: 4000 images
PolyU-Real-World-Noisy-Images-Dataset\train_sub\LQ: 4000 images

Total JPG images: 8000

EXAMPLE FILENAMES

PolyU-Real-World-Noisy-Images-Dataset\train_sub\GT
   Canon5D2_bag_s001.jpg
   Canon5D2_bag_s002.jpg
   Canon5D2_bag_s003.jpg

PolyU-Real-World-Noisy-Images-Dataset\train_sub\LQ
   Canon5D2_bag_s001.jpg
   Canon5D2_bag_s002.jpg
   Canon5D2_bag_s003.jpg


In [7]:

from pathlib import Path
import pandas as pd

# Locate the extracted PolyU dataset
polyu_base = (
    DATA_DIR
    / "PolyU"
    / "PolyU-Real-World-Noisy-Images-Dataset"
    / "train_sub"
)

gt_dir = polyu_base / "GT"
lq_dir = polyu_base / "LQ"

inventory_dir = ROOT / "data" / "inventory"
inventory_dir.mkdir(parents=True, exist_ok=True)

# Identify matching clean/noisy pairs
records = []

for clean_path in sorted(gt_dir.glob("*.jpg")):
    noisy_path = lq_dir / clean_path.name

    if not noisy_path.exists():
        continue

    # Example:
    # Canon5D2_bag_s001.jpg -> Canon5D2_bag
    # Keeps patches from the same source scene together
    scene_id = clean_path.stem.rsplit("_s", 1)[0]

    records.append({
        "noisy_path": noisy_path.relative_to(ROOT).as_posix(),
        "clean_path": clean_path.relative_to(ROOT).as_posix(),
        "group_id": scene_id
    })

df = pd.DataFrame(records)

output = inventory_dir / "polyu.csv"
df.to_csv(output, index=False)

print("Matched pairs:", len(df))
print("Unique scene groups:", df["group_id"].nunique())
print("Saved inventory:", output)

display(df.head())


Matched pairs: 4000
Unique scene groups: 40
Saved inventory: c:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\data\inventory\polyu.csv


,noisy_path,clean_path,group_id
0,data/raw/PolyU/PolyU-Real-World-Noisy-Images-D...,data/raw/PolyU/PolyU-Real-World-Noisy-Images-D...,Canon5D2_bag
1,data/raw/PolyU/PolyU-Real-World-Noisy-Images-D...,data/raw/PolyU/PolyU-Real-World-Noisy-Images-D...,Canon5D2_bag
2,data/raw/PolyU/PolyU-Real-World-Noisy-Images-D...,data/raw/PolyU/PolyU-Real-World-Noisy-Images-D...,Canon5D2_bag
3,data/raw/PolyU/PolyU-Real-World-Noisy-Images-D...,data/raw/PolyU/PolyU-Real-World-Noisy-Images-D...,Canon5D2_bag
4,data/raw/PolyU/PolyU-Real-World-Noisy-Images-D...,data/raw/PolyU/PolyU-Real-World-Noisy-Images-D...,Canon5D2_bag


In [8]:

from huggingface_hub import hf_hub_download
from pathlib import Path
import zipfile

REPO = "VCLab-PolyU/GGT-100K"

print("Downloading SIDD dataset...")

archive = hf_hub_download(
    repo_id=REPO,
    filename="Existing-dataset/SIDD.zip",
    repo_type="dataset"
)

destination = DATA_DIR / "SIDD"
destination.mkdir(parents=True, exist_ok=True)

print("Extracting SIDD...")

with zipfile.ZipFile(archive, "r") as z:
    z.extractall(destination)

print("SIDD extracted successfully!")
print("Location:", destination)


Extracting SIDD...
SIDD extracted successfully!
Location: c:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\data\raw\SIDD


In [9]:

from collections import Counter

sidd_root = DATA_DIR / "SIDD"

print("SIDD DATASET STRUCTURE")
print("=" * 50)

for folder in sorted(sidd_root.rglob("*")):
    if not folder.is_dir():
        continue

    counts = Counter(
        p.suffix.lower()
        for p in folder.iterdir()
        if p.is_file()
    )

    if counts:
        print(folder.relative_to(sidd_root), dict(counts))

print("\nSample image files:")

images = [
    p for p in sidd_root.rglob("*")
    if p.suffix.lower() in (".png", ".jpg", ".jpeg")
]

for image in images[:15]:
    print(image.relative_to(sidd_root))

print("\nTotal images:", len(images))


SIDD DATASET STRUCTURE
SIDD\train_sub512\GT {'.png': 16000}
SIDD\train_sub512\LQ {'.png': 16000}

Sample image files:
SIDD\train_sub512\GT\0001_001_S6_00100_00060_3200_L_010_s001.png
SIDD\train_sub512\GT\0001_001_S6_00100_00060_3200_L_010_s003.png
SIDD\train_sub512\GT\0001_001_S6_00100_00060_3200_L_010_s004.png
SIDD\train_sub512\GT\0001_001_S6_00100_00060_3200_L_010_s005.png
SIDD\train_sub512\GT\0001_001_S6_00100_00060_3200_L_010_s006.png
SIDD\train_sub512\GT\0001_001_S6_00100_00060_3200_L_010_s007.png
SIDD\train_sub512\GT\0001_001_S6_00100_00060_3200_L_010_s008.png
SIDD\train_sub512\GT\0001_001_S6_00100_00060_3200_L_010_s009.png
SIDD\train_sub512\GT\0001_001_S6_00100_00060_3200_L_010_s010.png
SIDD\train_sub512\GT\0001_001_S6_00100_00060_3200_L_010_s011.png
SIDD\train_sub512\GT\0001_001_S6_00100_00060_3200_L_010_s012.png
SIDD\train_sub512\GT\0001_001_S6_00100_00060_3200_L_010_s013.png
SIDD\train_sub512\GT\0001_001_S6_00100_00060_3200_L_010_s014.png
SIDD\train_sub512\GT\0001_001_S6_0010

In [10]:

from pathlib import Path
import pandas as pd

# Locate SIDD folders
sidd_base = DATA_DIR / "SIDD" / "SIDD" / "train_sub512"

gt_dir = sidd_base / "GT"
lq_dir = sidd_base / "LQ"

inventory_dir = ROOT / "data" / "inventory"
inventory_dir.mkdir(parents=True, exist_ok=True)

records = []

for clean_path in sorted(gt_dir.glob("*.png")):
    noisy_path = lq_dir / clean_path.name

    if not noisy_path.exists():
        continue

    # Remove patch suffix, e.g. _s001
    # All patches from the same original image stay together.
    scene_id = clean_path.stem.rsplit("_s", 1)[0]

    records.append({
        "noisy_path": noisy_path.relative_to(ROOT).as_posix(),
        "clean_path": clean_path.relative_to(ROOT).as_posix(),
        "group_id": scene_id
    })

df_sidd = pd.DataFrame(records)

output = inventory_dir / "sidd.csv"
df_sidd.to_csv(output, index=False)

print("Matched SIDD pairs:", len(df_sidd))
print("Unique scene groups:", df_sidd["group_id"].nunique())
print("Saved inventory:", output)

display(df_sidd.head())


Matched SIDD pairs: 16000
Unique scene groups: 320
Saved inventory: c:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\data\inventory\sidd.csv


,noisy_path,clean_path,group_id
0,data/raw/SIDD/SIDD/train_sub512/LQ/0001_001_S6...,data/raw/SIDD/SIDD/train_sub512/GT/0001_001_S6...,0001_001_S6_00100_00060_3200_L_010
1,data/raw/SIDD/SIDD/train_sub512/LQ/0001_001_S6...,data/raw/SIDD/SIDD/train_sub512/GT/0001_001_S6...,0001_001_S6_00100_00060_3200_L_010
2,data/raw/SIDD/SIDD/train_sub512/LQ/0001_001_S6...,data/raw/SIDD/SIDD/train_sub512/GT/0001_001_S6...,0001_001_S6_00100_00060_3200_L_010
3,data/raw/SIDD/SIDD/train_sub512/LQ/0001_001_S6...,data/raw/SIDD/SIDD/train_sub512/GT/0001_001_S6...,0001_001_S6_00100_00060_3200_L_010
4,data/raw/SIDD/SIDD/train_sub512/LQ/0001_001_S6...,data/raw/SIDD/SIDD/train_sub512/GT/0001_001_S6...,0001_001_S6_00100_00060_3200_L_010


In [11]:

print("SIDD INVENTORY VERIFICATION")
print("=" * 40)

print("Total matched pairs:", len(df_sidd))
print("Unique scene groups:", df_sidd["group_id"].nunique())

print("\nMissing values:")
print(df_sidd.isna().sum())

print("\nSample distribution per scene:")
display(df_sidd.groupby("group_id").size().head(10))

assert len(df_sidd) >= 3000, "Insufficient SIDD pairs!"

print("\nSIDD inventory is ready!")


SIDD INVENTORY VERIFICATION
Total matched pairs: 16000
Unique scene groups: 320

Missing values:
noisy_path    0
clean_path    0
group_id      0
dtype: int64

Sample distribution per scene:


group_id
0001_001_S6_00100_00060_3200_L_010    61
0001_001_S6_00100_00060_3200_L_011    60
0002_001_S6_00100_00020_3200_N_010    57
0002_001_S6_00100_00020_3200_N_011    61
0003_001_S6_00100_00060_3200_H_010    60
0003_001_S6_00100_00060_3200_H_011    60
0004_001_S6_00100_00060_4400_L_010    61
0004_001_S6_00100_00060_4400_L_011    63
0005_001_S6_00100_00060_4400_N_010    57
0005_001_S6_00100_00060_4400_N_011    62
dtype: int64


SIDD inventory is ready!


In [32]:

import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

inventory_dir = ROOT / "data" / "inventory"

sidd = pd.read_csv(inventory_dir / "sidd.csv")
polyu = pd.read_csv(inventory_dir / "polyu.csv")

print("DATASET INVENTORY")
print("=" * 40)

print("SIDD:", len(sidd), "pairs")
print("PolyU:", len(polyu), "pairs")

print("\nUnique source groups:")
print("SIDD:", sidd["group_id"].nunique())
print("PolyU:", polyu["group_id"].nunique())

assert len(sidd) >= 3500
assert len(polyu) >= 1500

print("\nBoth datasets are ready!")


DATASET INVENTORY
SIDD: 16000 pairs
PolyU: 4000 pairs

Unique source groups:
SIDD: 320
PolyU: 40

Both datasets are ready!


In [33]:

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

QUOTAS = {
    "SIDD": {"train": 2800, "val": 350, "test": 350},
    "PolyU": {"train": 1200, "val": 150, "test": 150}
}

def make_scene_groups(df, dataset):
    df = df.copy()

    if dataset == "SIDD":
        # Group captures sharing the leading scene ID.
        df["scene_id"] = (
            df["group_id"].astype(str)
            .str.split("_").str[0]
        )
    else:
        # PolyU group_id already excludes patch suffix.
        df["scene_id"] = df["group_id"].astype(str)

    return df

def allocate_dataset(df, dataset, quotas):
    df = make_scene_groups(df, dataset)

    groups = df["scene_id"].unique().tolist()
    rng.shuffle(groups)

    total = len(groups)

    if total < 3:
        raise ValueError("Not enough independent groups")

    # Assign complete scenes to partitions.
    # Approximate allocation, then sample exact quotas.
    train_end = int(total * 0.8)
    val_end = int(total * 0.9)

    group_sets = {
        "train": set(groups[:train_end]),
        "val": set(groups[train_end:val_end]),
        "test": set(groups[val_end:])
    }

    outputs = []

    for split, group_set in group_sets.items():
        subset = df[df["scene_id"].isin(group_set)]

        required = quotas[split]

        if len(subset) < required:
            raise ValueError(
                f"{dataset} {split}: "
                f"Need {required} samples, "
                f"only {len(subset)} available. "
                "Adjust scene allocation."
            )

        selected = subset.sample(
            n=required,
            random_state=SEED
        ).copy()

        selected["dataset"] = dataset
        selected["split"] = split

        outputs.append(selected)

    return pd.concat(outputs, ignore_index=True)

sidd_selected = allocate_dataset(
    sidd, "SIDD", QUOTAS["SIDD"]
)

polyu_selected = allocate_dataset(
    polyu, "PolyU", QUOTAS["PolyU"]
)

splits = pd.concat(
    [sidd_selected, polyu_selected],
    ignore_index=True
)

# Shuffle the final manifest
splits = splits.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

output = ROOT / "data" / "splits.csv"
splits.to_csv(output, index=False)

print("Saved:", output)
print("Total samples:", len(splits))

display(pd.crosstab(
    splits["dataset"],
    splits["split"]
))


Saved: c:\Users\rmclo\OneDrive\Documents\GitHub\CCS250_CompVis_Final\data\splits.csv
Total samples: 5000


split,test,train,val
dataset,,,
PolyU,150,1200,150
SIDD,350,2800,350


In [35]:

print("FINAL DATASET VERIFICATION")
print("=" * 45)

# Check total
assert len(splits) == 5000

# Verify split sizes
counts = splits["split"].value_counts()

assert counts["train"] == 4000
assert counts["val"] == 500
assert counts["test"] == 500

# Verify no duplicated image pairs
assert not splits.duplicated(
    subset=["dataset", "noisy_path", "clean_path"]
).any()

# Verify scene-level separation
for dataset in ["SIDD", "PolyU"]:
    part = splits[splits["dataset"] == dataset]

    scene_sets = {
        split: set(
            part.loc[
                part["split"] == split,
                "scene_id"
            ]
        )
        for split in ["train", "val", "test"]
    }

    assert scene_sets["train"].isdisjoint(scene_sets["val"])
    assert scene_sets["train"].isdisjoint(scene_sets["test"])
    assert scene_sets["val"].isdisjoint(scene_sets["test"])

print("All validation checks passed!")

print("\nFinal distribution:")
display(pd.crosstab(
    splits["dataset"],
    splits["split"],
    margins=True
))


FINAL DATASET VERIFICATION
All validation checks passed!

Final distribution:


split,test,train,val,All
dataset,,,,
PolyU,150,1200,150,1500
SIDD,350,2800,350,3500
All,500,4000,500,5000


In [36]:

missing = []

for _, row in splits.iterrows():
    for col in ["noisy_path", "clean_path"]:
        path = ROOT / row[col]

        if not path.is_file():
            missing.append(str(path))

print("Missing image files:", len(missing))

if missing:
    print("First missing paths:")
    for p in missing[:5]:
        print(p)
else:
    print("All 5,000 noisy/clean pairs are accessible!")


Missing image files: 0
All 5,000 noisy/clean pairs are accessible!


In [37]:

import pandas as pd
from pathlib import Path
from PIL import Image

splits = pd.read_csv(ROOT / "data" / "splits.csv")

# Check exact distribution
expected = {
    ("SIDD", "train"): 2800,
    ("SIDD", "val"): 350,
    ("SIDD", "test"): 350,
    ("PolyU", "train"): 1200,
    ("PolyU", "val"): 150,
    ("PolyU", "test"): 150
}

counts = splits.groupby(["dataset", "split"]).size()

for key, target in expected.items():
    assert counts.get(key, 0) == target, f"Incorrect allocation: {key}"

# Check scene leakage
for dataset, subset in splits.groupby("dataset"):
    scene_sets = {
        split: set(part["scene_id"])
        for split, part in subset.groupby("split")
    }

    for a, b in [
        ("train", "val"),
        ("train", "test"),
        ("val", "test")
    ]:
        assert scene_sets[a].isdisjoint(scene_sets[b]), (
            f"Scene leakage detected: {dataset}, {a}/{b}"
        )

# Verify representative image pairs
for _, row in splits.groupby(
    ["dataset", "split"]
).head(3).iterrows():

    with Image.open(ROOT / row["noisy_path"]) as noisy:
        noisy_size = noisy.size

    with Image.open(ROOT / row["clean_path"]) as clean:
        clean_size = clean.size

    assert noisy_size == clean_size, (
        f"Image size mismatch: {row['noisy_path']}"
    )

print("Dataset allocation: PASSED")
print("Scene separation: PASSED")
print("Sample image dimensions: PASSED")
print("\nReady for training!")


Dataset allocation: PASSED
Scene separation: PASSED
Sample image dimensions: PASSED

Ready for training!


In [38]:

from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import torch

# 1. Locate project
ROOT = Path.cwd()
if not (ROOT / "app").is_dir():
    ROOT = ROOT.parent

# 2. Load finalized dataset manifest
manifest = ROOT / "data" / "splits.csv"
df = pd.read_csv(manifest)

# 3. Verify allocation
expected = {
    ("SIDD", "train"): 2800,
    ("SIDD", "val"): 350,
    ("SIDD", "test"): 350,
    ("PolyU", "train"): 1200,
    ("PolyU", "val"): 150,
    ("PolyU", "test"): 150,
}

for (source, split), count in expected.items():
    actual = len(df[
        (df["dataset"] == source) &
        (df["split"] == split)
    ])
    assert actual == count, f"{source}/{split}: {actual} != {count}"

# 4. Load one actual noisy/clean pair
sample = df[df["split"] == "train"].iloc[0]

with Image.open(ROOT / sample["noisy_path"]) as img:
    noisy = np.array(img.convert("RGB"))

with Image.open(ROOT / sample["clean_path"]) as img:
    clean = np.array(img.convert("RGB"))

assert noisy.shape == clean.shape, "Image dimensions mismatch!"

# 5. Convert to PyTorch tensors
noisy_tensor = torch.from_numpy(
    noisy.copy()
).permute(2, 0, 1).float() / 255.0

clean_tensor = torch.from_numpy(
    clean.copy()
).permute(2, 0, 1).float() / 255.0

# 6. Verify GPU tensor loading
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

noisy_tensor = noisy_tensor.to(device)
clean_tensor = clean_tensor.to(device)

print("=" * 50)
print("iMAGE DATASET — FINAL READINESS CHECK")
print("=" * 50)

print("Total samples:", len(df))
print("Training:", (df["split"] == "train").sum())
print("Validation:", (df["split"] == "val").sum())
print("Testing:", (df["split"] == "test").sum())

print("\nSample source:", sample["dataset"])
print("Image shape:", noisy.shape)
print("Tensor shape:", tuple(noisy_tensor.shape))
print("Tensor device:", noisy_tensor.device)

print("\nDataset manifest: PASSED")
print("Image loading: PASSED")
print("Tensor conversion: PASSED")
print("GPU transfer:", "PASSED" if device.type == "cuda" else "CPU ONLY")

print("\nREADY FOR MODEL TRAINING!")


iMAGE DATASET — FINAL READINESS CHECK
Total samples: 5000
Training: 4000
Validation: 500
Testing: 500

Sample source: SIDD
Image shape: (512, 512, 3)
Tensor shape: (3, 512, 512)
Tensor device: cuda:0

Dataset manifest: PASSED
Image loading: PASSED
Tensor conversion: PASSED
GPU transfer: PASSED

READY FOR MODEL TRAINING!
